# Esophageal Cancer Research - Clinical Table
* By Sangwon Baek
* Samsung Medical Center
* August 3rd, 2023

### Import necessary packages and read data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from statsmodels.graphics.mosaicplot import mosaic
import itertools

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.options.mode.chained_assignment = None

In [2]:
df = pd.read_csv("../Data/Preprocessed/ECA_Dataset.csv").drop(columns="Unnamed: 0")

### Create Clinical Table

In [88]:
# Replace NaN values in 'Primary_Site' with 'mid'
df['Primary_Site'].fillna('mid', inplace=True)

# Select the specified columns
selected_columns = ["pTNM7_1", "Primary_Site", "total_pos_LN", "total_LN", "Sex", "Age", "Operation_Date"]
df_selected = df[selected_columns]

# Correct the error with Primary Site to correct value

df_selected.isna().any()

pTNM7_1           False
Primary_Site      False
total_pos_LN      False
total_LN          False
Sex               False
Age               False
Operation_Date    False
dtype: bool

In [89]:
def N_categorize(x):
    if x == 0:
        return '0'
    elif 1 <= x <= 2:
        return '1'
    elif 3 <= x <= 6:
        return '2'
    else: # x > 7
        return '3'

In [90]:
# Extract T, N, and M categories using regular expressions and then remove the prefixes
df_selected['T_category'] = df_selected['pTNM7_1'].str.extract('(TX|T0|Tis|T1a|T1b|T2|T3|T4a|T4b)').replace('T', '', regex=True)
df_selected['N_category'] = df_selected.total_pos_LN.apply(N_categorize)
df_selected['M_category'] = df_selected['pTNM7_1'].str.extract('(M0|M1)').replace('M', '', regex=True)

# Filtering out 'is' category and creating the two groups
T1_group = df_selected[df_selected['T_category'].isin(['1a', '1b'])]
T2_4_Group = df_selected[df_selected['T_category'].isin(['2', '3', '4a', '4b'])]

In [91]:
original_N_values = df.total_pos_LN.apply(N_categorize).value_counts().to_list()
recalculated_N_values = df.total_pos_LN_new.apply(N_categorize).value_counts().to_list()

# Creating the dataframe
N_df = pd.DataFrame({
    'Original': original_N_values,
    'Without Others': recalculated_N_values
}, index=['N=0', 'N=1', 'N=2', 'N=3'])

N_df

,Original,Without Others
N=0,1482,1509
N=1,655,661
N=2,368,353
N=3,158,140


In [92]:
df_selected

,pTNM7_1,Primary_Site,total_pos_LN,total_LN,Sex,Age,Operation_Date,T_category,N_category,M_category
0,pT3N2M0,mid,4,69,M,62.0,1993-09-23,3,2,0
1,pT2N1M0,lower,1,29,M,50.0,1993-10-13,2,1,0
2,pT3N1M0,lower,2,38,M,54.0,1993-11-24,3,1,0
3,pT3N0M0,mid,0,36,M,52.0,1993-12-19,3,0,0
4,pT2N3M0,mid,8,32,M,71.0,1994-04-26,2,3,0
...,...,...,...,...,...,...,...,...,...,...
2658,pT1bN0M0,lower,0,5,M,72.0,2021-12-26,1b,0,0
2659,pT1bN1M0,lower,1,9,M,69.0,2021-12-27,1b,1,0
2660,pT3N0M0,lower,0,28,M,61.0,2021-12-27,3,0,0
2661,pT1bN0M0,lower,0,29,M,66.0,2021-12-12,1b,0,0


In [93]:
df_selected.total_LN

0       69
1       29
2       38
3       36
4       32
        ..
2658     5
2659     9
2660    28
2661    29
2662    26
Name: total_LN, Length: 2663, dtype: int64

In [94]:
def characteristics_table(group):
    """Extract and format baseline characteristics for a given group."""
    # Calculate combined count and percentage for Tis and T1a
    combined_count = group['T_category'].value_counts().get('1a', 0) + group['T_category'].value_counts().get('Tis', 0)
    combined_percentage = combined_count / len(group) * 100
    
    data = {
        'Age': f"{group['Age'].mean():.2f} ({group['Age'].std():.2f})",
        'Male Sex': f"{group['Sex'].value_counts().get('M', 0):,.0f} ({group['Sex'].value_counts(normalize=True).get('M', 0) * 100:.1f}%)",
        'Total Dissected LN': f"{group['total_LN'].mean():.2f} ({group['total_LN'].std():.2f})",
        'Primary Site Upper': f"{group['Primary_Site'].value_counts().get('upper', 0):,.0f} ({group['Primary_Site'].value_counts(normalize=True).get('upper', 0) * 100:.1f}%)",
        'Primary Site Mid': f"{group['Primary_Site'].value_counts().get('mid', 0):,.0f} ({group['Primary_Site'].value_counts(normalize=True).get('mid', 0) * 100:.1f}%)",
        'Primary Site Lower': f"{group['Primary_Site'].value_counts().get('lower', 0):,.0f} ({group['Primary_Site'].value_counts(normalize=True).get('lower', 0) * 100:.1f}%)",
        'pTis&pT1a': f"{combined_count:,.0f} ({combined_percentage:.1f}%)",
        'pT1b': f"{group['T_category'].value_counts().get('1b', 0):,.0f} ({group['T_category'].value_counts(normalize=True).get('1b', 0) * 100:.1f}%)",
        'pT2': f"{group['T_category'].value_counts().get('2', 0):,.0f} ({group['T_category'].value_counts(normalize=True).get('2', 0) * 100:.1f}%)",
        'pT3': f"{group['T_category'].value_counts().get('3', 0):,.0f} ({group['T_category'].value_counts(normalize=True).get('3', 0) * 100:.1f}%)",
        'pT4a': f"{group['T_category'].value_counts().get('4a', 0):,.0f} ({group['T_category'].value_counts(normalize=True).get('4a', 0) * 100:.1f}%)",
        'pT4b': f"{group['T_category'].value_counts().get('4b', 0):,.0f} ({group['T_category'].value_counts(normalize=True).get('4b', 0) * 100:.1f}%)",
        'pNx': f"{group['N_category'].value_counts().get('x', 0):,.0f} ({group['N_category'].value_counts(normalize=True).get('x', 0) * 100:.1f}%)",
        'pN0': f"{group['N_category'].value_counts().get('0', 0):,.0f} ({group['N_category'].value_counts(normalize=True).get('0', 0) * 100:.1f}%)",
        'pN1': f"{group['N_category'].value_counts().get('1', 0):,.0f} ({group['N_category'].value_counts(normalize=True).get('1', 0) * 100:.1f}%)",
        'pN2': f"{group['N_category'].value_counts().get('2', 0):,.0f} ({group['N_category'].value_counts(normalize=True).get('2', 0) * 100:.1f}%)",
        'pN3': f"{group['N_category'].value_counts().get('3', 0):,.0f} ({group['N_category'].value_counts(normalize=True).get('3', 0) * 100:.1f}%)",
        # 'pM0': f"{group['M_category'].value_counts().get('0', 0):,.0f} ({group['M_category'].value_counts(normalize=True).get('0', 0) * 100:.1f}%)",
        # 'pM1': f"{group['M_category'].value_counts().get('1', 0):,.0f} ({group['M_category'].value_counts(normalize=True).get('1', 0) * 100:.1f}%)"
    }
    return data

def compute_cohens_h(P_Counts, P_total, N_Counts, N_total):
    """Compute Cohen's H given counts and total numbers."""
    P_proportion = P_Counts / P_total
    N_proportion = N_Counts / N_total
    
    h = 2 * np.arcsin(np.sqrt(P_proportion)) - 2 * np.arcsin(np.sqrt(N_proportion))
    return round(abs(h), 3)

def compute_cohens_d(P_mean, P_std, P_total, N_mean, N_std, N_total):
    """Compute Cohen's D for continuous data given means, standard deviations, and total numbers."""
    pooled_std = np.sqrt(((N_total-1)*(N_std**2) + (P_total-1)*(P_std**2))/(N_total+P_total-2))
    d = (P_mean - N_mean) / pooled_std
    return round(abs(d), 3)

def compute_ASMD(group_1, group_1_data, group_2, group_2_data):
    # Re-calculate Cohen's D for Age and Cohen's H for other characteristics
    ASMD = {}
    for key in group_1_data.keys():
        # For Age and Dissected Lymph Nodes compute Cohen's D for continuous data
        if key == "Age" or key == "Total Dissected LN":
            P_mean = float(group_1_data[key].split()[0])
            N_mean = float(group_2_data[key].split()[0])
            P_std = float(group_1_data[key].split()[1].strip('()'))
            N_std = float(group_2_data[key].split()[1].strip('()'))
            ASMD[key] = compute_cohens_d(P_mean, P_std, len(group_1), N_mean, N_std, len(group_2))
        # For other characteristics, compute Cohen's H for categorical data
        else:
            P_value = int(group_1_data[key].split()[0].replace(',', ''))
            N_value = int(group_2_data[key].split()[0].replace(',', ''))
            ASMD[key] = compute_cohens_h(P_value, len(group_1), N_value, len(group_2))
    return ASMD


In [95]:
T1_group

,pTNM7_1,Primary_Site,total_pos_LN,total_LN,Sex,Age,Operation_Date,T_category,N_category,M_category
6,pT1bN0M0,lower,0,1,M,67.0,1994-06-22,1b,0,0
14,pT1bN1M0,lower,1,35,M,45.0,1994-11-16,1b,1,0
15,pT1bN0M0,mid,0,31,M,59.0,1994-11-23,1b,0,0
16,pT1bN1M0,mid,1,32,M,59.0,1994-11-27,1b,1,0
19,pT1bN0M0,lower,0,23,M,51.0,1995-03-05,1b,0,0
...,...,...,...,...,...,...,...,...,...,...
2657,pT1bN0M0,mid,0,19,F,85.0,2021-12-23,1b,0,0
2658,pT1bN0M0,lower,0,5,M,72.0,2021-12-26,1b,0,0
2659,pT1bN1M0,lower,1,9,M,69.0,2021-12-27,1b,1,0
2661,pT1bN0M0,lower,0,29,M,66.0,2021-12-12,1b,0,0


In [96]:
# Extract data for both groups using the function
T1_group_data = characteristics_table(T1_group[T1_group.N_category!='0'])
T2_4_group_data = characteristics_table(T2_4_Group[T2_4_Group.N_category!='0'])

ASMD_result = compute_ASMD(T1_group, T1_group_data, T2_4_Group, T2_4_group_data)

# Updating the column names to include total counts
column_names = {
    'T_1': f"pT1 (N = {len(T1_group[T1_group.N_category!='0'])})",
    'T_2_4': f"pT2-4 (N = {len(T2_4_Group[T2_4_Group.N_category!='0'])})"
}

# Compile into a DataFrame
clinical_table_df = pd.DataFrame({
    column_names['T_1']: T1_group_data,
    column_names['T_2_4']: T2_4_group_data,
    "ASMD": ASMD_result
})

In [97]:
nan_rows = T2_4_Group[T2_4_Group['Primary_Site'].isna()]
nan_rows

,pTNM7_1,Primary_Site,total_pos_LN,total_LN,Sex,Age,Operation_Date,T_category,N_category,M_category


In [98]:
T2_4_Group[(T2_4_Group['N_category'] != '0') & (T2_4_Group['Primary_Site'] == 'upper')]


,pTNM7_1,Primary_Site,total_pos_LN,total_LN,Sex,Age,Operation_Date,T_category,N_category,M_category
11,pT3N2M0,upper,4,60,M,74.0,1994-11-01,3,2,0
17,pT3N1M0,upper,1,21,M,64.0,1994-12-18,3,1,0
18,pT3N1M0,upper,1,95,M,72.0,1995-01-03,3,1,0
53,pT3N3M0,upper,7,105,M,63.0,1996-03-13,3,3,0
99,pT3N3M0,upper,10,54,M,69.0,1997-08-06,3,3,0
103,pT3N3M0,upper,7,82,M,71.0,1997-10-19,3,3,0
121,pT3N3M0,upper,10,61,M,43.0,1998-02-08,3,3,0
198,pT3N1M0,upper,17,195,M,58.0,1999-08-30,3,3,0
206,pT3N1M0,upper,2,78,M,59.0,1999-11-10,3,1,0
218,pT3N3M0,upper,9,87,M,62.0,2000-01-09,3,3,0


In [99]:
clinical_table_df.to_csv("../Results/clinical_table.csv")
clinical_table_df

,pT1 (N = 391),pT2-4 (N = 790),ASMD
Age,62.50 (8.15),64.82 (8.40),0.281
Male Sex,370 (94.6%),754 (95.4%),0.801
Total Dissected LN,37.67 (14.44),44.71 (19.41),0.418
Primary Site Upper,47 (12.0%),82 (10.4%),0.174
Primary Site Mid,193 (49.4%),374 (47.3%),0.454
Primary Site Lower,151 (38.6%),334 (42.3%),0.469
pTis&pT1a,37 (9.5%),0 (0.0%),0.319
pT1b,354 (90.5%),0 (0.0%),1.027
pT2,0 (0.0%),195 (24.7%),0.838
pT3,0 (0.0%),538 (68.1%),1.483


In [100]:
# Extract T, N, and M categories using regular expressions and then remove the prefixes
df['T_category'] = df['pTNM7_1'].str.extract('(TX|T0|Tis|T1a|T1b|T2|T3|T4a|T4b)').replace('T', '', regex=True)
df['N_category'] = df.total_pos_LN.apply(N_categorize)
df['M_category'] = df['pTNM7_1'].str.extract('(M0|M1)').replace('M', '', regex=True)

# Filtering out 'is' category and creating the two groups
T1_df = df[df['T_category'].isin(['1a', '1b'])]
T24_df = df[df['T_category'].isin(['2', '3', '4a', '4b'])]

#Create the subgroup dfs for the subgroup analysis
T1_upper_df = T1_df.loc[T1_df.Primary_Site=='upper']
T1_mid_df = T1_df.loc[T1_df.Primary_Site=='mid']
T1_lower_df = T1_df.loc[T1_df.Primary_Site=='lower']

T24_upper_df = T24_df.loc[T24_df.Primary_Site=='upper']
T24_mid_df = T24_df.loc[T24_df.Primary_Site=='mid']
T24_lower_df = T24_df.loc[T24_df.Primary_Site=='lower']

pos_columns = [col for col in df.columns if col.startswith("pos_")]


In [101]:
# Define a function to calculate both the sum and proportion for a given DataFrame
def calculate_counts_and_proportions(df):
    # Ensure that each row of pos_columns has a value of 1
    df_clipped = df[pos_columns].clip(upper=1)
    
    # Calculate the counts and convert to integer
    counts = df_clipped.sum().astype(int)
    
    # Calculate proportions based on the subgroup size, round to 2 decimals
    proportions = (counts / df_clipped.shape[0] * 100).round(2)
    
    return counts, proportions

# Initialize a dictionary to store the results
results_dict = {}

# Make sure all N_category are strings.
T1_upper_df.N_category.astype(str)
T24_upper_df.N_category.astype(str)
T1_mid_df.N_category.astype(str)
T24_mid_df.N_category.astype(str)
T1_lower_df.N_category.astype(str)
T24_lower_df.N_category.astype(str)

# Loop through each subgroup DataFrame
subgroup_dfs = {
    'T1_upper': T1_upper_df[T1_upper_df.N_category!='0'],
    'T24_upper': T24_upper_df[T24_upper_df.N_category!='0'],
    'T1_mid': T1_mid_df[T1_mid_df.N_category!='0'],
    'T24_mid': T24_mid_df[T24_mid_df.N_category!='0'],
    'T1_lower': T1_lower_df[T1_lower_df.N_category!='0'],
    'T24_lower': T24_lower_df[T24_lower_df.N_category!='0']
}

# Modify the loop to include the count of each subgroup in the column header
for group_name, subgroup_df in subgroup_dfs.items():
    # Append the count of the subgroup to the group name
    header_with_count = f"{group_name} (N={subgroup_df.shape[0]})"
    
    counts, proportions = calculate_counts_and_proportions(subgroup_df)
    
    # Convert counts and proportions into the desired format
    combined_results = [f"{c} ({p}%)"
                        for c, p in zip(counts, proportions)]
    
    # Store the combined results in the dictionary using the modified header as the key
    results_dict[header_with_count] = combined_results

# List of index names to remove
rows_to_remove = [
    'pos_ON', 'pos_neckLN', 'pos_OM', 'pos_mediaLN', 'pos_OA',
    'pos_abdoLN', 'pos_neckLN_new', 'pos_mediaLN_new', 'pos_abdoLN_new'
]

# Convert the results dictionary to a DataFrame
results_df = pd.DataFrame(results_dict, index=pos_columns)

# Drop those rows
results_df = results_df.drop(rows_to_remove, errors='ignore')  # errors='ignore' ensures it won't fail if any index is not present

# Save the DataFrame to a CSV file
results_df.to_csv("../Results/frequency_table.csv")


In [102]:
T24_mid_df[T24_mid_df.N_category!='0']

,Patient_Number,Sex,Age,Year,Operation_Date,IC_cm,lesion_text,Lesion_Coding,pos_101R,total_101R,pos_101L,total_101L,pos_102R,total_102R,pos_102L,total_102L,pos_104R,total_104R,pos_104L,total_104L,other_text_neck,pos_ON,total_ON,pos_neckLN,total_neckLN,pos_106preR,total_106preR,pos_106preL,total_106preL,pos_106recR,total_106recR,pos_106recL,total_106recL,pos_107,total_107,pos_105/108/110,total_105/108/110,pos_112pulR,total_112pulR,pos_112pulL,total_112pulL,other_text_media,pos_OM,total_OM,pos_mediaLN,total_mediaLN,pos_1/2/7,total_1/2/7,pos_8,total_8,pos_9,total_9,other_text_abdo,pos_OA,total_OA,pos_abdoLN,total_abdoLN,total_pos_LN,total_LN,pTNM7_1,pos_neckLN_new,pos_mediaLN_new,pos_abdoLN_new,total_pos_LN_new,Primary_Site,T_category,N_category,M_category
0,n_000045,M,62.0,94.0,1993-09-23,30.0,mid,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,2,31,0,1,0,0,"""mediastinal LN"",0/9;",0.0,9.0,2,41,2,20,0,8,0,0,NaN,NaN,NaN,2,28,4,69,pT3N2M0,0,2,2,4,mid,3,2,0
4,n_000144,M,71.0,95.0,1994-04-26,"28-33, 40",mid,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,7,2,5,0,0,0,0,NaN,NaN,NaN,2,12,6,12,0,3,0,5,NaN,NaN,NaN,6,20,8,32,pT2N3M0,0,2,6,8,mid,2,3,0
8,n_000286,M,62.0,95.0,1994-09-07,31-37,mid,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,"""Lt. jugular"", 0/7; ""anterior portion of neck""...",4.0,34.0,4,34,0,0,0,0,0,0,0,0,0,0,1,14,0,0,0,0,"""high medial LN"", 0/4; ""paratracheal, subcarin...",0.0,39.0,1,53,0,31,0,0,0,1,NaN,NaN,NaN,0,32,5,119,pT2N2M0,0,1,0,1,mid,2,2,0
28,n_000596,M,59.0,96.0,1995-05-10,30-35,mid,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,NaN,NaN,NaN,0,0,0,0,0,10,0,0,0,0,0,0,0,15,0,0,0,0,NaN,NaN,NaN,0,25,0,0,0,4,2,12,NaN,NaN,NaN,2,16,2,41,pT3N1M0,0,0,2,2,mid,3,1,0
31,n_000689,F,69.0,96.0,1995-07-09,"22,28",mid,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,NaN,NaN,NaN,0,0,2,7,0,0,0,0,0,0,0,0,1,1,0,0,0,0,"""high mediastinal LN"", 1/5;",1.0,5.0,4,13,0,12,0,5,0,1,"""Greater curvature"", 0/1;",0.0,1.0,0,19,4,32,pT3N2M0,0,3,0,3,mid,3,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2598,n_050182,M,66.0,22.0,2021-03-22,32-34,Thoracic-middle,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0.0,0.0,0,0,0,0,0,0,0,2,0,8,0,3,0,2,0,3,0,2,0.0,0.0,0.0,0,20,1,13,0,1,0,2,0.0,0.0,0.0,1,16,1,36,pT2N1M0,0,0,1,1,mid,2,1,0
2601,n_050243,M,76.0,22.0,2021-03-29,25.0,Thoracic-middle,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0.0,0.0,0,0,0,0,0,0,0,2,0,6,0,7,0,4,0,0,0,0,0.0,0.0,0.0,0,19,1,6,0,4,0,2,0.0,0.0,0.0,1,12,1,31,pT2N1M0,0,0,1,1,mid,2,1,0
2614,n_050630,M,63.0,22.0,2021-05-13,28-36,Thoracic-middle,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0.0,0.0,0,0,0,0,0,0,2,3,0,5,0,10,0,4,0,0,0,0,0.0,0.0,0.0,2,22,0,6,0,0,0,1,0.0,0.0,0.0,0,7,2,29,pT2N1M0,0,2,0,2,mid,2,1,0
2630,n_051220,M,65.0,22.0,2021-07-22,25-29,Thoracic-middle,2.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,1,5,2,4,0.0,0.0,0.0,4,12,0,0,0,0,0,3,3,7,0,7,0,0,0,0,0,0,0.0,0.0,0.0,3,17,0,3,0,4,0,0,0.0,0.0,0.0,0,7,7,36,pT3N3M0,4,3,0,7,mid,3,3,0


In [103]:
results_df

,T1_upper (N=47),T24_upper (N=82),T1_mid (N=193),T24_mid (N=374),T1_lower (N=151),T24_lower (N=334)
pos_101R,3 (6.38%),7 (8.54%),4 (2.07%),11 (2.94%),1 (0.66%),1 (0.3%)
pos_101L,1 (2.13%),8 (9.76%),1 (0.52%),15 (4.01%),0 (0.0%),2 (0.6%)
pos_102R,0 (0.0%),2 (2.44%),0 (0.0%),1 (0.27%),0 (0.0%),0 (0.0%)
pos_102L,0 (0.0%),1 (1.22%),1 (0.52%),1 (0.27%),0 (0.0%),0 (0.0%)
pos_104R,1 (2.13%),9 (10.98%),1 (0.52%),10 (2.67%),0 (0.0%),2 (0.6%)
pos_104L,4 (8.51%),7 (8.54%),0 (0.0%),5 (1.34%),0 (0.0%),2 (0.6%)
pos_106preR,0 (0.0%),0 (0.0%),1 (0.52%),7 (1.87%),0 (0.0%),2 (0.6%)
pos_106preL,0 (0.0%),0 (0.0%),1 (0.52%),12 (3.21%),0 (0.0%),5 (1.5%)
pos_106recR,24 (51.06%),45 (54.88%),66 (34.2%),139 (37.17%),27 (17.88%),64 (19.16%)
pos_106recL,13 (27.66%),40 (48.78%),49 (25.39%),109 (29.14%),22 (14.57%),42 (12.57%)
